In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
df=pd.read_csv("houses_messy.csv")

In [5]:
#price_k去除k
df_clean=df.copy()
df_clean['price_k']=(
    df_clean['price_k']
    .astype(str)
    .str.strip()
    .str.replace('k','',case=False,regex=False)
)
df_clean['price_k']=pd.to_numeric(df_clean['price_k'],errors='coerce')
#统一neighborhood格式
df_clean['neighborhood']=(
    df_clean['neighborhood']
    .astype('string')
    .str.strip()
    .str.lower()
)
#去除离群数据
df_clean.loc[df_clean['age_years']>100,'age_years']=np.nan
df_clean=df_clean.dropna(subset=['price_k'])
q1=df_clean['price_k'].quantile(0.25)
q3=df_clean['price_k'].quantile(0.75)
iqr=q3-q1
lower=q1-1.5*iqr
upper=q3+1.5*iqr
df_clean=df_clean[
    (df_clean['price_k']>=lower)&
    (df_clean['price_k']<=upper)
]
#去除空值
num_cols=[
    'area_sqm',
    'bedrooms',
    'bathrooms',
    'age_years',
    'distance_to_center_km'
]
for col in num_cols:
    df_clean[col]=df_clean[col].fillna(df_clean[col].median())
df_clean['neighborhood']=df_clean['neighborhood'].fillna(
    df_clean['neighborhood'].mode()[0]
)
#独热编码
df_model=df_clean.copy()
df_model=pd.get_dummies(df_clean,columns=['neighborhood'],dtype=int,drop_first=True)
X = df_model.drop("price_k", axis=1).values
Y = df_model["price_k"].values

# 记录特征名，之后看系数用
feature_names = df_model.drop("price_k", axis=1).columns.tolist()

In [7]:
df_model

,area_sqm,bedrooms,bathrooms,age_years,has_garage,distance_to_center_km,price_k,neighborhood_greenfield,neighborhood_hillview,neighborhood_oldtown,neighborhood_riverside
0,120.7,3,1.0,45.0,1,8.9,444.5,0,0,0,1
1,73.6,2,1.0,5.0,1,3.6,251.0,0,0,1,0
2,136.3,4,2.0,35.0,1,13.3,410.6,0,0,0,0
3,142.9,4,2.0,20.0,0,10.1,507.3,1,0,0,0
4,41.7,1,1.0,22.0,1,15.7,160.5,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...
1195,178.6,4,3.0,36.0,1,7.0,561.3,0,0,1,0
1196,67.0,2,1.0,46.0,0,4.6,195.3,1,0,0,0
1197,77.6,2,1.0,7.0,1,10.2,298.5,0,1,0,0
1198,109.8,4,2.0,2.0,0,1.3,654.2,0,0,1,0


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [11]:
X_train,X_test,Y_train,Y_test=train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42
)
X_train.shape


    

(944, 10)

In [13]:
model=LinearRegression()
model.fit(X_train,Y_train)
Y_pred=model.predict(X_test)
mae=mean_absolute_error(Y_test,Y_pred)
rmse=np.sqrt(mean_squared_error(Y_test,Y_pred))
r2=r2_score(Y_test,Y_pred)

In [14]:
print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

MAE: 32.95829160362489
RMSE: 43.55581953983467
R2: 0.8891848985869286


In [15]:
coef_df = pd.DataFrame({
    "feature": feature_names,
    "coef": model.coef_
})

coef_df

,feature,coef
0,area_sqm,3.198086
1,bedrooms,22.625677
2,bathrooms,3.725260
3,age_years,-1.807063
4,has_garage,14.361382
5,distance_to_center_km,-4.503898
6,neighborhood_greenfield,10.841380
7,neighborhood_hillview,80.768937
8,neighborhood_oldtown,53.922870
9,neighborhood_riverside,107.725181


In [16]:
model.intercept_

-3.4680731738816917